# Box comparison cis25 vs cis100 — statistical samples with cosmic-variance errors

**Question**: can the surface-density / attenuation-gradient study survive at m100n1024
resolution — and how much cosmic variance does each box carry?

**Design** (supersedes the earlier 1-object-per-bin version of this notebook):

1. **Selection** directly from the caesar catalogs of both boxes at the *same* snapshots
   (both sims share the output-time list): snap 134 (z≈0.3) and snap 105 (z≈1).
   Two classes — **quenched** (sSFR < 0.2/t_H) and **star-forming** (sSFR ≥ 0.5/t_H) —
   in stellar-mass bins; green-valley objects in between are excluded.
2. **Number densities** per (box, class, mass bin) from the *full* selection, with
   delete-one-octant jackknife errors → the direct cosmic-variance statistic.
3. **Reduced particle files** (the lean `build_reduced_particles_job.py` product:
   gas m/dust/HI/H2/sfr/temp + stars, 100 kpc aperture incl. CGM, stellar principal
   frame) for a capped random subsample per cell — the notebook writes the SLURM plan,
   `submit_reduced_particles.sh` builds the files (idempotent: galaxies already extracted
   for other campaigns are skipped).
4. **Stacked face-on Σ\*(R), Σ_dust(R)** medians per cell with two error terms:
   bootstrap over galaxies (object scatter) and octant jackknife (cosmic variance).

Runs on the cluster (`pd39` kernel). Created 2026-08-05.

In [ ]:
# --- Part 0: configuration -------------------------------------------------
import os, json, warnings
import numpy as np
import h5py
import matplotlib.pyplot as plt
from astropy.table import Table
from astropy.cosmology import FlatLambdaCDM
import astropy.units as u

HOME = '/mnt/home/glorenzon/analize_simba_cgm'
SIMS = ('cis25', 'cis100')
ANCHORS = (134, 105)                     # same snap <-> z mapping in both boxes

with open(os.path.expanduser('~/.simbanator/config.json')) as f:
    _SIMCFG = json.load(f)['simulations']

def sim_prefix(sim):
    return _SIMCFG[sim]['file_format'].split('_{')[0]      # m25n512 / m100n1024

def caesar_file(sim, snap):
    c = _SIMCFG[sim]
    return os.path.join(c['catalog_dir'], c['file_format'].format(snap=snap))

# selection
LOGM_MIN, NSTAR_MIN = 10.0, 20
MASS_BINS = [(10.0, 10.4), (10.4, 10.8), (10.8, 11.6)]
PASSIVE_FACTOR = 0.2      # quenched: sSFR <  0.2 / t_H(z)
SF_FACTOR = 0.5           # SF:       sSFR >= 0.5 / t_H(z)   (between the two = green valley, dropped)
CENTRALS_ONLY = False     # satellites carry part of the environment difference between boxes

# extraction subsample (number densities always use the FULL selection)
CAP_PER_CELL, SEED = 60, 42

# cosmic-variance jackknife: box divided into N_JK_SIDE^3 sub-volumes
N_JK_SIDE = 2

# measurement
ANNULI_KPC = np.logspace(np.log10(0.5), np.log10(30.0), 11)   # physical kpc, face-on
APERTURES_KPC = [1.0, 3.162, 10.0, 31.62]
STAR_MEMBER_ONLY = False   # False: all stars in the 100 kpc reduced aperture (same in both boxes)

OUTDIR = f'{HOME}/output/box_resolution'
FIGDIR = f'{OUTDIR}/figures'
os.makedirs(FIGDIR, exist_ok=True)

COLORS = {'cis25': '#0072B2', 'cis100': '#D55E00'}            # Okabe-Ito, CVD-safe
BOXLAB = {'cis25': 'cis25 (m25n512)', 'cis100': 'cis100 (m100n1024)'}
CLASSES = ('Q', 'SF')
CLSLAB = {'Q': 'quenched', 'SF': 'star-forming'}

def reduced_path(sim, snap, gx):
    return (f'{HOME}/output/{sim}/reduced_particles/snap_{int(snap):03d}/'
            f'{sim_prefix(sim)}_snap{int(snap):03d}_gal{int(gx):06d}.h5')

def bin_label(lo, hi):
    return f'{lo:g}-{hi:g}'


## Part 1 — selection from the caesar catalogs

Direct `h5py` reads (no caesar import): `galaxy_data` scalars + `dicts/masses.stellar`.
Positions are comoving kpc (`kpccm`, h-free) and give each galaxy its jackknife
sub-volume. Classes use sSFR relative to 1/t_H(z) with the thresholds from Part 0.

In [ ]:
# --- Part 1: build the full samples ----------------------------------------
def read_cat(sim, snap):
    with h5py.File(caesar_file(sim, snap), 'r') as f:
        sa = f['simulation_attributes'].attrs
        gd = f['galaxy_data']
        d = dict(
            z=float(sa['redshift']), h=float(sa['hubble_constant']),
            om=float(sa['omega_matter']), box_ckpc=float(sa['boxsize']),
            a=float(sa['scale_factor']),
            gid=gd['GroupID'][:], pos=gd['pos'][:],                # kpccm (h-free)
            sfr=gd['sfr'][:], nstar=gd['nstar'][:], ngas=gd['ngas'][:],
            central=gd['central'][:].astype(bool),
            mstar=gd['dicts/masses.stellar'][:],
            mdust=gd['dicts/masses.dust'][:],
            r50=gd['dicts/radii.stellar_half_mass'][:],
        )
    return d

def octant_of(pos, box_ckpc):
    cell = box_ckpc / N_JK_SIDE
    ijk = np.clip((pos // cell).astype(int), 0, N_JK_SIDE - 1)
    return ijk[:, 0] * N_JK_SIDE**2 + ijk[:, 1] * N_JK_SIDE + ijk[:, 2]

META, rows = {}, []
for sim in SIMS:
    for snap in ANCHORS:
        d = read_cat(sim, snap)
        cosmo = FlatLambdaCDM(H0=100 * d['h'], Om0=d['om'])
        tH_yr = cosmo.age(d['z']).to(u.yr).value
        META[(sim, snap)] = dict(z=d['z'], tH_yr=tH_yr,
                                 vol_cMpc3=(d['box_ckpc'] / 1e3)**3, a=d['a'])
        with np.errstate(divide='ignore', invalid='ignore'):
            logm = np.log10(np.maximum(d['mstar'], 1.0))
            ssfr = d['sfr'] / np.maximum(d['mstar'], 1.0)
        base = (logm >= LOGM_MIN) & (d['nstar'] >= NSTAR_MIN)
        if CENTRALS_ONLY:
            base &= d['central']
        cls = np.where(ssfr < PASSIVE_FACTOR / tH_yr, 'Q',
                       np.where(ssfr >= SF_FACTOR / tH_yr, 'SF', 'GV'))
        octs = octant_of(d['pos'], d['box_ckpc'])
        n_gv = int(np.sum(base & (cls == 'GV')))
        for i in np.flatnonzero(base & (cls != 'GV')):
            lohi = [b for b in MASS_BINS if b[0] <= logm[i] < b[1]]
            if not lohi:
                continue
            rows.append(dict(sim=sim, snap=int(snap), gal_id=int(d['gid'][i]),
                             cls=str(cls[i]), massbin=bin_label(*lohi[0]),
                             logm=float(logm[i]), ssfr=float(ssfr[i]),
                             nstar=int(d['nstar'][i]), ngas=int(d['ngas'][i]),
                             central=bool(d['central'][i]), oct=int(octs[i]),
                             logmdust=float(np.log10(max(d['mdust'][i], 1.0))),
                             r50_kpc=float(d['r50'][i] * d['a'])))
        print(f"[{sim} snap {snap} z={d['z']:.3f}] selected "
              f"Q={np.sum(base & (cls=='Q'))} SF={np.sum(base & (cls=='SF'))} "
              f"(GV dropped: {n_gv})")

SAMPLE = Table(rows=rows)
SAMPLE.write(f'{OUTDIR}/boxcomp_sample.fits', overwrite=True)
len(SAMPLE)


In [ ]:
# --- Part 1b: number densities with octant-jackknife errors ----------------
def jackknife(values_per_unit, estimator):
    '''Delete-one jackknife over sub-volume units. values_per_unit: dict oct -> data;
    estimator(list_of_units_kept) -> scalar. Returns (theta_all, sigma_jk).'''
    keys = sorted(values_per_unit)
    theta_all = estimator(keys)
    if len(keys) < 2:
        return theta_all, np.nan
    thetas = np.array([estimator([k for k in keys if k != kk]) for kk in keys])
    K = len(keys)
    return theta_all, np.sqrt((K - 1) / K * np.sum((thetas - thetas.mean())**2))

DENS_ROWS = []
K_TOT = N_JK_SIDE**3
for sim in SIMS:
    for snap in ANCHORS:
        V = META[(sim, snap)]['vol_cMpc3']
        for cls in CLASSES:
            for lo, hi in MASS_BINS:
                lab = bin_label(lo, hi)
                sub = SAMPLE[(SAMPLE['sim'] == sim) & (SAMPLE['snap'] == snap)
                             & (SAMPLE['cls'] == cls) & (SAMPLE['massbin'] == lab)]
                cnt = {k: int(np.sum(sub['oct'] == k)) for k in range(K_TOT)}
                def _n(kept, cnt=cnt, V=V):
                    return sum(cnt[k] for k in kept) / (V * len(kept) / K_TOT)
                n, s = jackknife(cnt, _n)
                DENS_ROWS.append(dict(sim=sim, snap=int(snap), cls=cls, massbin=lab,
                                      N=len(sub), n_cMpc3=n, sigma_jk=s,
                                      rel_err=(s / n if n > 0 else np.nan)))

DENS = Table(rows=DENS_ROWS)
DENS.write(f'{OUTDIR}/boxcomp_number_densities.fits', overwrite=True)
DENS['sim', 'snap', 'cls', 'massbin', 'N', 'n_cMpc3', 'rel_err']


## Part 2 — extraction plan for the reduced particle files

A capped random subsample per (box, anchor, class, mass bin) goes into one SLURM plan
per box (the schema `build_reduced_particles_job.py` reads: `sim_name` attr +
`entry_gx`/`entry_snap`). The job is idempotent — galaxies already extracted for other
campaigns (e.g. the quench-mode anchors) are skipped on sight. After writing the plans,
submit on the cluster:

```bash
cd /mnt/home/glorenzon/analize_simba_cgm && mkdir -p logs
for s in cis25 cis100; do
  DUST_PLAN=output/$s/caesar_sfh/prof_boxcomp/dust_profile_plan_boxcomp.hdf5 \
    sbatch --array=0-1 submit_reduced_particles.sh
done
```

(2 array tasks = one per anchor snapshot; raise `--mem` if an m100 snapshot pass swaps.)
Then re-run Part 3.

In [ ]:
# --- Part 2: subsample + plan files + coverage -----------------------------
rng = np.random.default_rng(SEED)
keep_idx = []
for sim in SIMS:
    for snap in ANCHORS:
        for cls in CLASSES:
            for lo, hi in MASS_BINS:
                idx = np.flatnonzero((SAMPLE['sim'] == sim) & (SAMPLE['snap'] == snap)
                                     & (SAMPLE['cls'] == cls)
                                     & (SAMPLE['massbin'] == bin_label(lo, hi)))
                if len(idx) > CAP_PER_CELL:
                    idx = rng.choice(idx, CAP_PER_CELL, replace=False)
                keep_idx.extend(idx.tolist())

EXTRACT = SAMPLE[sorted(keep_idx)]
EXTRACT.write(f'{OUTDIR}/boxcomp_extract_sample.fits', overwrite=True)

for sim in SIMS:
    sub = EXTRACT[EXTRACT['sim'] == sim]
    plan_dir = f'{HOME}/output/{sim}/caesar_sfh/prof_boxcomp'
    os.makedirs(plan_dir, exist_ok=True)
    plan = f'{plan_dir}/dust_profile_plan_boxcomp.hdf5'
    with h5py.File(plan, 'w') as f:
        f.attrs['sim_name'] = sim
        f.attrs['note'] = 'box_resolution_comparison.ipynb statistical sample'
        f.create_dataset('entry_gx', data=np.asarray(sub['gal_id'], np.int64))
        f.create_dataset('entry_snap', data=np.asarray(sub['snap'], np.int32))
    have = sum(os.path.exists(reduced_path(sim, r['snap'], r['gal_id'])) for r in sub)
    print(f"[{sim}] plan: {len(sub)} galaxies -> {plan}")
    print(f"        reduced files already on disk: {have}/{len(sub)}")


## Part 3 — measure the reduced files

Face-on (stellar principal frame from the reduced file's `evecs`) Σ\*(R) and Σ_dust(R)
on the Part-0 annuli, plus star/gas particle counts inside the projected RT apertures.
Missing files are reported and skipped, so this cell can run before/while the SLURM job
completes and be re-run afterwards.

In [ ]:
# --- Part 3: per-galaxy measurements ---------------------------------------
NB_R = len(ANNULI_KPC) - 1
AREA_PC2 = np.pi * (ANNULI_KPC[1:]**2 - ANNULI_KPC[:-1]**2) * 1e6

def measure_reduced(sim, snap, gx):
    p = reduced_path(sim, snap, gx)
    if not os.path.exists(p):
        return None
    try:
        with h5py.File(p, 'r') as f:
            ev = np.asarray(f.attrs['evecs'], float)
            out = {}
            for grp, mkey in (('star', 'm_star'), ('gas', 'm_dust')):
                g = f.get(grp)
                if g is None or 'pos' not in g or len(g['pos']) == 0:
                    out[grp] = (np.zeros(0), np.zeros(0), np.zeros(0, bool))
                    continue
                proj = np.asarray(g['pos'][:], float) @ ev.T
                R = np.hypot(proj[:, 0], proj[:, 1])
                m = np.asarray(g[mkey][:], float) if mkey in g else np.zeros(len(R))
                mem = (np.asarray(g['member'][:], bool) if 'member' in g
                       else np.ones(len(R), bool))
                out[grp] = (R, m, mem)
    except (OSError, RuntimeError) as e:
        print(f'  [corrupt] {os.path.basename(p)}: {e}')
        return None
    Rs, ms, mems = out['star']
    if STAR_MEMBER_ONLY:
        Rs, ms = Rs[mems], ms[mems]
    Rg, md, _ = out['gas']
    res = dict(
        sig_star=np.histogram(Rs, bins=ANNULI_KPC, weights=ms)[0] / AREA_PC2,
        sig_dust=np.histogram(Rg, bins=ANNULI_KPC, weights=md)[0] / AREA_PC2,
        n_star=np.histogram(Rs, bins=ANNULI_KPC)[0].astype(float),
        nstar_ap=np.array([np.sum(Rs < ap) for ap in APERTURES_KPC], float),
        ngas_ap=np.array([np.sum(Rg < ap) for ap in APERTURES_KPC], float),
    )
    return res

M = dict(sig_star=[], sig_dust=[], n_star=[], nstar_ap=[], ngas_ap=[], row=[])
n_missing = {}
for i, r in enumerate(EXTRACT):
    res = measure_reduced(r['sim'], r['snap'], r['gal_id'])
    if res is None:
        k = (r['sim'], int(r['snap']))
        n_missing[k] = n_missing.get(k, 0) + 1
        continue
    for k in ('sig_star', 'sig_dust', 'n_star', 'nstar_ap', 'ngas_ap'):
        M[k].append(res[k])
    M['row'].append(i)

MROW = EXTRACT[M.pop('row')]
M = {k: np.array(v) for k, v in M.items()}
print(f"measured {len(MROW)}/{len(EXTRACT)} galaxies; missing per (sim, snap): "
      f"{n_missing or 'none'}")

with h5py.File(f'{OUTDIR}/boxcomp_measurements.hdf5', 'w') as f:
    f.attrs['annuli_kpc'] = ANNULI_KPC
    f.attrs['apertures_kpc'] = APERTURES_KPC
    for k, v in M.items():
        f.create_dataset(k, data=v, compression='lzf')
    for c in ('sim', 'snap', 'gal_id', 'cls', 'massbin', 'oct', 'logm'):
        v = np.asarray(MROW[c])
        f.create_dataset(f'meta/{c}', data=v.astype('S') if v.dtype.kind == 'U' else v)


## Part 4 — stacked profiles with object-scatter and cosmic-variance errors

Per (box, anchor, class, mass bin): the **median** profile over galaxies, with

- **σ_boot** — bootstrap over galaxies (error of the median from object-to-object scatter);
- **σ_JK** — delete-one-octant jackknife (adds the large-scale-structure / cosmic-variance
  term; this is what the small box cannot beat by resolution).

Figures show the median with the JK band (outer, light) and bootstrap band (inner);
the error-budget table quantifies σ_JK/σ_boot.

In [ ]:
# --- Part 4: stacking + error decomposition --------------------------------
RMID = np.sqrt(ANNULI_KPC[:-1] * ANNULI_KPC[1:])
N_BOOT = 400
_srng = np.random.default_rng(SEED + 1)

def stack_cell(sim, snap, cls, lab, key):
    sel = ((MROW['sim'] == sim) & (MROW['snap'] == snap)
           & (MROW['cls'] == cls) & (MROW['massbin'] == lab))
    A = M[key][np.asarray(sel)]
    if len(A) < 3:
        return None
    med = np.median(A, axis=0)
    boots = np.median(A[_srng.integers(0, len(A), (N_BOOT, len(A)))], axis=1)
    sig_boot = np.std(boots, axis=0)
    octs = np.asarray(MROW['oct'][np.asarray(sel)])
    occupied = np.unique(octs)
    if len(occupied) >= 2:
        jk = np.array([np.median(A[octs != k], axis=0) for k in occupied])
        K = len(occupied)
        sig_jk = np.sqrt((K - 1) / K * np.sum((jk - jk.mean(axis=0))**2, axis=0))
    else:
        sig_jk = np.full(med.shape, np.nan)
    return dict(n=len(A), med=med, sig_boot=sig_boot, sig_jk=sig_jk)

STACKS, budget_rows = {}, []
for sim in SIMS:
    for snap in ANCHORS:
        for cls in CLASSES:
            for lo, hi in MASS_BINS:
                lab = bin_label(lo, hi)
                for key in ('sig_star', 'sig_dust', 'nstar_ap', 'ngas_ap'):
                    st = stack_cell(sim, snap, cls, lab, key)
                    STACKS[(sim, snap, cls, lab, key)] = st
                    if st is not None and key in ('sig_star', 'sig_dust'):
                        with np.errstate(invalid='ignore', divide='ignore'):
                            ratio = np.nanmedian(st['sig_jk'] / st['sig_boot'])
                            rel = np.nanmedian(st['sig_jk'] / np.abs(st['med']))
                        budget_rows.append(dict(sim=sim, snap=int(snap), cls=cls,
                                                massbin=lab, key=key, n_gal=st['n'],
                                                jk_over_boot=float(ratio),
                                                jk_rel=float(rel)))

BUDGET = Table(rows=budget_rows)
BUDGET.write(f'{OUTDIR}/boxcomp_error_budget.fits', overwrite=True)
print(BUDGET)


In [ ]:
# --- Part 4b: profile figures (one per anchor x class) ----------------------
def _prof_panel(ax, snap, cls, lab, key, ylab, last_row):
    for sim in SIMS:
        st = STACKS[(sim, snap, cls, lab, key)]
        if st is None:
            continue
        ok = st['med'] > 0
        ax.plot(RMID[ok], st['med'][ok], color=COLORS[sim], lw=1.8, marker='o',
                ms=3.5, label=f"{BOXLAB[sim]} (n={st['n']})")
        lo_jk = np.maximum(st['med'] - st['sig_jk'], 0)
        ax.fill_between(RMID[ok], lo_jk[ok], (st['med'] + st['sig_jk'])[ok],
                        color=COLORS[sim], alpha=0.12, lw=0)
        lo_b = np.maximum(st['med'] - st['sig_boot'], 0)
        ax.fill_between(RMID[ok], lo_b[ok], (st['med'] + st['sig_boot'])[ok],
                        color=COLORS[sim], alpha=0.25, lw=0)
    ax.set(xscale='log', yscale='log', ylabel=ylab)
    if last_row:
        ax.set_xlabel('R  [pkpc, face-on]')
    ax.grid(alpha=0.25, which='both', lw=0.5)
    ax.set_title(f'log M* = {lab}', fontsize=10)

for snap in ANCHORS:
    z = META[(SIMS[0], snap)]['z']
    for cls in CLASSES:
        fig, axes = plt.subplots(len(MASS_BINS), 2,
                                 figsize=(9, 3.1 * len(MASS_BINS)),
                                 sharex=True, squeeze=False)
        for i, (lo, hi) in enumerate(MASS_BINS):
            last = i == len(MASS_BINS) - 1
            _prof_panel(axes[i, 0], snap, cls, bin_label(lo, hi), 'sig_star',
                        r'$\Sigma_\ast$  [M$_\odot$ pc$^{-2}$]', last)
            _prof_panel(axes[i, 1], snap, cls, bin_label(lo, hi), 'sig_dust',
                        r'$\Sigma_{\rm dust}$  [M$_\odot$ pc$^{-2}$]', last)
        if axes[0, 0].get_legend_handles_labels()[0]:
            axes[0, 0].legend(frameon=False, fontsize=8, loc='lower left')
        fig.suptitle(f'{CLSLAB[cls]}, z = {z:.2f} — median; bands: bootstrap (inner), '
                     f'octant JK / cosmic variance (outer)', y=1.005, fontsize=11)
        fig.tight_layout()
        for ext in ('png', 'pdf'):
            fig.savefig(f'{FIGDIR}/stack_profiles_snap{snap}_{cls}.{ext}',
                        dpi=200, bbox_inches='tight')
        plt.show()


In [ ]:
# --- Part 4c: number densities + aperture sampling --------------------------
XB = np.arange(len(MASS_BINS))
BINLABS = [bin_label(*b) for b in MASS_BINS]

for snap in ANCHORS:
    z = META[(SIMS[0], snap)]['z']
    fig, axes = plt.subplots(1, 2, figsize=(9, 3.4), sharey=True)
    for ax, cls in zip(axes, CLASSES):
        for j, sim in enumerate(SIMS):
            sub = DENS[(DENS['sim'] == sim) & (DENS['snap'] == snap)
                       & (DENS['cls'] == cls)]
            y = np.array([sub[sub['massbin'] == lb]['n_cMpc3'][0] for lb in BINLABS])
            e = np.array([sub[sub['massbin'] == lb]['sigma_jk'][0] for lb in BINLABS])
            ax.errorbar(XB + (j - 0.5) * 0.12, np.maximum(y, 1e-8), yerr=e,
                        fmt='o', ms=6, lw=1.8, capsize=3, color=COLORS[sim],
                        label=BOXLAB[sim])
        ax.set(yscale='log', title=f'{CLSLAB[cls]}, z = {z:.2f}',
               xticks=XB, xticklabels=BINLABS, xlabel='log M*')
        ax.grid(axis='y', alpha=0.25, lw=0.5)
    axes[0].set_ylabel(r'n  [cMpc$^{-3}$]')
    axes[1].legend(frameon=False, fontsize=9)
    fig.tight_layout()
    for ext in ('png', 'pdf'):
        fig.savefig(f'{FIGDIR}/number_densities_snap{snap}.{ext}',
                    dpi=200, bbox_inches='tight')
    plt.show()

# aperture sampling: median counts with 16-84% intervals over each cell's galaxies
for snap in ANCHORS:
    z = META[(SIMS[0], snap)]['z']
    fig, axes = plt.subplots(len(CLASSES), 2, figsize=(9.5, 6.4), sharey=True,
                             squeeze=False)
    XA = np.arange(len(APERTURES_KPC))
    for ci, cls in enumerate(CLASSES):
        for ax, key, ttl in ((axes[ci, 0], 'nstar_ap', 'stars'),
                             (axes[ci, 1], 'ngas_ap', 'gas')):
            for j, sim in enumerate(SIMS):
                sel = ((MROW['sim'] == sim) & (MROW['snap'] == snap)
                       & (MROW['cls'] == cls))
                A = M[key][np.asarray(sel)]
                if len(A) < 3:
                    continue
                q16, q50, q84 = np.percentile(A, [16, 50, 84], axis=0)
                off = (j - 0.5) * 0.24
                ax.errorbar(XA + off, np.maximum(q50, 0.5),
                            yerr=[np.maximum(q50 - q16, 0), np.maximum(q84 - q50, 0)],
                            fmt='o', ms=6, lw=1.8, capsize=3, color=COLORS[sim],
                            label=BOXLAB[sim] if (ci, ttl) == (0, 'gas') else None)
            ax.axhline(100, color='0.4', ls=':', lw=1)
            ax.set(yscale='log', title=f'{ttl}, {CLSLAB[cls]}, z = {z:.2f}',
                   xticks=XA, xticklabels=[f'<{a:g} kpc' for a in APERTURES_KPC])
            ax.grid(axis='y', alpha=0.25, lw=0.5)
        axes[ci, 0].set_ylabel('particles in aperture')
    if axes[0, 1].get_legend_handles_labels()[0]:
        axes[0, 1].legend(frameon=False, fontsize=9, loc='upper left')
    fig.tight_layout()
    for ext in ('png', 'pdf'):
        fig.savefig(f'{FIGDIR}/aperture_sampling_stat_snap{snap}.{ext}',
                    dpi=200, bbox_inches='tight')
    plt.show()


## Part 5 — reading the result

Fill in after the SLURM extraction completes and Parts 3–4 are re-run. What to look at:

- **`boxcomp_number_densities.fits` / the density figure** — `rel_err` is the fractional
  cosmic variance per cell. Expect the cis25 quenched high-mass cells to be both sparse
  (small N) and strongly octant-dependent; that is the honest cost of the small box.
- **`boxcomp_error_budget.fits`** — `jk_over_boot` > 1 means cosmic variance dominates the
  stacked-profile error for that cell; `jk_rel` is the total fractional band on the median.
- **Aperture sampling** — the per-galaxy particle counts inside 1–32 kpc apertures set the
  shot-noise floor of any per-aperture attenuation/Σ measurement; compare the gas panels
  of the two boxes for the quenched class in particular.

**Caveats**
1. Both classes share one selection machinery, but the reduced files include halo/CGM
   material inside 100 kpc for *both* boxes (candidate set = parent-halo lists) — a fair,
   identical treatment, unlike the earlier region-vs-plist comparison.
2. The octant jackknife with 8 sub-volumes (12.5 vs 50 cMpc/h cells) underestimates
   variance on scales larger than the box — for cis25 the true cosmic variance is a
   *lower bound*.
3. Profiles are face-on medians; no inclination or kernel-smoothing correction. Dust Σ in
   quenched cis100 cells rests on few particles per galaxy — the stack does not remove
   that; see the aperture-sampling figure.
4. Green-valley galaxies are excluded from both classes by construction.